# Ingestão de Boletos com PySpark
This notebook uses **PySpark** to ingest, inspect, clean, and explore `base_boletos_fiap.csv`.

## 1. Imports and start SparkSession


In [29]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName('Ingestao Boletos')
    .config('spark.driver.memory', '2g')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel('WARN')
print(f'Spark version: {spark.version}')

Spark version: 4.1.1


## 2. Define Schema & Ingest CSV
Defining the schema explicitly avoids a full scan for type inference and makes ingestion faster and more reliable.

In [30]:
FILE_PATH = '../../input_data/base_boletos_fiap.csv'


schema = StructType([
    StructField('id_boleto',       StringType(), nullable=True),
    StructField('id_pagador',      StringType(), nullable=True),
    StructField('id_beneficiario', StringType(), nullable=True),
    StructField('dt_emissao',      StringType(), nullable=True),
    StructField('dt_vencimento',   StringType(), nullable=True),
    StructField('dt_pagamento',    StringType(), nullable=True),
    StructField('vlr_nominal',     DoubleType(), nullable=True),
    StructField('vlr_baixa',       DoubleType(), nullable=True),
    StructField('tipo_baixa',      StringType(), nullable=True),
    StructField('tipo_especie',    StringType(), nullable=True),
])

df_raw = (
    spark.read
    .option('header', 'true')
    .schema(schema)
    .csv(FILE_PATH)
)

print(f'Rows ingested: {df_raw.count():,}')
print(f'Columns      : {len(df_raw.columns)}')
df_raw.printSchema()

Rows ingested: 7,118
Columns      : 10
root
 |-- id_boleto: string (nullable = true)
 |-- id_pagador: string (nullable = true)
 |-- id_beneficiario: string (nullable = true)
 |-- dt_emissao: string (nullable = true)
 |-- dt_vencimento: string (nullable = true)
 |-- dt_pagamento: string (nullable = true)
 |-- vlr_nominal: double (nullable = true)
 |-- vlr_baixa: double (nullable = true)
 |-- tipo_baixa: string (nullable = true)
 |-- tipo_especie: string (nullable = true)



In [31]:
df_raw.describe().show()


+-------+--------------------+--------------------+--------------------+----------+-------------+------------+------------------+------------------+--------------------+-----------------+
|summary|           id_boleto|          id_pagador|     id_beneficiario|dt_emissao|dt_vencimento|dt_pagamento|       vlr_nominal|         vlr_baixa|          tipo_baixa|     tipo_especie|
+-------+--------------------+--------------------+--------------------+----------+-------------+------------+------------------+------------------+--------------------+-----------------+
|  count|                7118|                7118|                7118|      7118|         7118|        7048|              7118|              6298|                7048|             7118|
|   mean|                NULL|                NULL|                NULL|      NULL|         NULL|        NULL|23300.574331272837| 20570.83138297873|                NULL|             NULL|
| stddev|                NULL|                NULL|         

In [32]:
df_raw.select('dt_emissao').distinct().orderBy('dt_emissao').show(20, truncate=False)

+----------+
|dt_emissao|
+----------+
|2019-08-16|
|2021-01-29|
|2021-03-18|
|2021-04-28|
|2021-06-18|
|2021-07-02|
|2021-07-12|
|2021-07-30|
|2021-08-30|
|2021-10-29|
|2021-12-15|
|2021-12-22|
|2022-02-09|
|2022-04-27|
|2022-06-20|
|2022-06-22|
|2022-06-23|
|2022-06-30|
|2022-07-20|
|2022-07-28|
+----------+
only showing top 20 rows


In [33]:
df_raw.filter(F.col('dt_emissao') == '2021-07-12').show(10, truncate=False)

+----------------------------------------------------------------+----------------------------------------------------------------+----------------------------------------------------------------+----------+-------------+------------+-----------+---------+--------------------------------+----------------------+
|id_boleto                                                       |id_pagador                                                      |id_beneficiario                                                 |dt_emissao|dt_vencimento|dt_pagamento|vlr_nominal|vlr_baixa|tipo_baixa                      |tipo_especie          |
+----------------------------------------------------------------+----------------------------------------------------------------+----------------------------------------------------------------+----------+-------------+------------+-----------+---------+--------------------------------+----------------------+
|294fe7b9e70f5aac70e41ee849e8a8dd40e9ed4ee304326e176cd03384a9

In [34]:
df_raw.select('dt_emissao').distinct().count()

250

In [35]:
BRONZE_PATH = '../../output_data/bronze/boletos'

df_raw = df_raw.withColumn('ingestion_timestamp', F.current_timestamp()) \
        .withColumn('partition_date', F.current_date())

df_raw.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').partitionBy('partition_date').save(BRONZE_PATH)
print(f'Bronze layer saved to {BRONZE_PATH}')

Bronze layer saved to ../../output_data/bronze/boletos


In [36]:
df_bronze = spark.read.format('delta').load(BRONZE_PATH)
print(f'Rows in bronze layer: {df_bronze.count():,}')
df_bronze.printSchema()

Rows in bronze layer: 7,118
root
 |-- id_boleto: string (nullable = true)
 |-- id_pagador: string (nullable = true)
 |-- id_beneficiario: string (nullable = true)
 |-- dt_emissao: string (nullable = true)
 |-- dt_vencimento: string (nullable = true)
 |-- dt_pagamento: string (nullable = true)
 |-- vlr_nominal: double (nullable = true)
 |-- vlr_baixa: double (nullable = true)
 |-- tipo_baixa: string (nullable = true)
 |-- tipo_especie: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- partition_date: date (nullable = true)



In [37]:
df_bronze.show(5, truncate=False)

+----------------------------------------------------------------+----------------------------------------------------------------+----------------------------------------------------------------+----------+-------------+------------+-----------+---------+---------------------------------------------+---------------------------------+--------------------------+--------------+
|id_boleto                                                       |id_pagador                                                      |id_beneficiario                                                 |dt_emissao|dt_vencimento|dt_pagamento|vlr_nominal|vlr_baixa|tipo_baixa                                   |tipo_especie                     |ingestion_timestamp       |partition_date|
+----------------------------------------------------------------+----------------------------------------------------------------+----------------------------------------------------------------+----------+-------------+------------+--------